# Transitivité stochastique

In [10]:
import pandas as pd
import numpy as np

## Concept

Si P(i bat j) ≥ 0.5 et P(j bat k) ≥ 0.5, alors on doit avoir P(i bat k) ≥ max(P(i bat j), P(j bat k))

Le problème c'est : comment savoir si la différence de score β entre le modèle rang 3 et le modèle rang 5 est réelle ou juste due au hasard ?

Le test de puissance répond exactement à ça : quel est le nombre minimum de comparaisons nécessaire pour détecter une différence réelle entre rang 3 et rang 5, avec 80% de chances de la détecter (puissance) et moins de 5% de faux positifs (α = 0.05) ?

β₃ et β₅ les scores des deux modèles
La différence β₃ - β₅ est l'effet que l'on cherches à détecter

Plus cette différence est petite, plus il faut de comparaisons pour la détecter statistiquement.

Test de puissance pour une proportion :

n = (z_α + z_β)² / (p₁ - p₂)²

Où p₁ et p₂ sont les probabilités de victoire dérivées des β via Bradley-Terry.

Et

p₃₅ = 1 / (1 + exp(-(β₃ - β₅)))
C'est simplement la fonction sigmoïde appliquée à la différence des β.


Le calcul de puissance donne un résultat du nombre minimum de comparaisons nécessaire.

Si les modèles rang 3 et rang 5 ont bien été comparés au moins tant de fois : le classement est statistiquement fiable à ces rangs.
Si non : impossible de distinguer le rang 3 du rang 5 avec confiance.

Les 20 modèles sont le filtre préalable pour s'assurer que nous travaillons uniquement sur des modèles qui ont assez de comparaisons pour que le test soit valide.

### Etapes

* Filtrer sur les 20 modèles les plus comparés
* Estimer les β sur ce sous-ensemble depuis le classement global (exercice 2.1)
* Calculer le nombre minimum de comparaisons nécessaire (calcul de puissance sur p₃₅ = σ(β₃ - β₅) et p₅₃ = 1 - p₃₅)
* Vérifier que rang 3 et rang 5 ont bien ce minimum dans tes données

In [ ]:
df = pd.read_json("../data/comparia-votes/votes_samples.jsonl", lines=True)

comparisons = pd.concat([
    df['model_a_name'], 
    df['model_b_name']
]).value_counts()

top20 = comparisons.head(20).index.tolist()

df_top20 = df[
    df['model_a_name'].isin(top20) & 
    df['model_b_name'].isin(top20)
]

In [20]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import binom

# ── 0. Chargement ─────────────────────────────────────────────────────────────
df = pd.read_json("../data/comparia-votes/votes_samples.jsonl", lines=True)

COL_A      = "model_a_name"
COL_B      = "model_b_name"
COL_WINNER = "chosen_model_name"

# Nettoyage préventif
df[COL_A]      = df[COL_A].str.strip()
df[COL_B]      = df[COL_B].str.strip()
df[COL_WINNER] = df[COL_WINNER].str.strip()

# Vérification rapide
print("Valeurs uniques de chosen_model_name :", df[COL_WINNER].value_counts().head(10))

# ── ÉTAPE 1 : Sous-ensemble top 20 + fit Bradley-Terry ───────────────────────
top20_models = (
    pd.concat([df[COL_A], df[COL_B]])
    .value_counts()
    .head(20)
    .index.tolist()
)

df_top20 = df[
    df[COL_A].isin(top20_models) & df[COL_B].isin(top20_models)
].copy()

print(f"\nNombre de votes dans le top20 : {len(df_top20)}")


def fit_bradley_terry(df_pairs, col_winner=COL_WINNER, col_a=COL_A, col_b=COL_B):
    models = sorted(set(df_pairs[col_a]) | set(df_pairs[col_b]))
    idx = {m: i for i, m in enumerate(models)}
    n = len(models)

    wins = np.zeros((n, n))
    for _, row in df_pairs.iterrows():
        w = row[col_winner]
        a, b = row[col_a], row[col_b]
        if w == a:
            wins[idx[a], idx[b]] += 1
        elif w == b:
            wins[idx[b], idx[a]] += 1
        # both_equal / NaN → ignoré dans BT de base

    def neg_log_lik(beta):
        nll = 0.0
        for i in range(n):
            for j in range(n):
                if wins[i, j] > 0:
                    nll -= wins[i, j] * np.log(expit(beta[i] - beta[j]) + 1e-12)
        return nll

    beta0  = np.zeros(n)
    bounds = [(0, 0)] + [(None, None)] * (n - 1)
    res    = minimize(neg_log_lik, beta0, method="L-BFGS-B", bounds=bounds)

    result = (
        pd.DataFrame({"model": models, "beta": res.x})
        .sort_values("beta", ascending=False)
        .reset_index(drop=True)
    )
    result["rank"] = result.index + 1
    return result


ranking = fit_bradley_terry(df_top20)
print("\nClassement global (top 20) :")
print(ranking[["rank", "model", "beta"]].to_string(index=False))

beta3 = ranking.loc[ranking["rank"] == 3, "beta"].values[0]
beta5 = ranking.loc[ranking["rank"] == 5, "beta"].values[0]
model3 = ranking.loc[ranking["rank"] == 3, "model"].values[0]
model5 = ranking.loc[ranking["rank"] == 5, "model"].values[0]

print(f"\nRang 3 : {model3}  β = {beta3:.4f}")
print(f"Rang 5 : {model5}  β = {beta5:.4f}")


# ── ÉTAPE 2 : Calcul de puissance → n_min ────────────────────────────────────
def min_comparisons_power(beta_i, beta_j, alpha=0.05, power=0.80):
    p_ij = expit(beta_i - beta_j)
    print(f"\np35 = σ(β3 − β5) = {p_ij:.4f}")
    print(f"p53 = 1 − p35    = {1 - p_ij:.4f}")

    for n in range(1, 10_001):
        k_crit = binom.ppf(1 - alpha, n, 0.5)
        pw     = 1 - binom.cdf(k_crit - 1, n, p_ij)
        if pw >= power:
            return n, int(k_crit), p_ij, pw

    return None, None, p_ij, None


n_min, k_crit, p_ij, achieved_power = min_comparisons_power(beta3, beta5)
print(f"\nn_min          = {n_min}")
print(f"Seuil rejet    = {k_crit} victoires")
print(f"Puissance réelle = {achieved_power:.3f}")

# ── ÉTAPE 3 : Vérifier que rang 3 vs rang 5 ont ≥ n_min comparaisons ─────────
mask = (
    ((df[COL_A] == model3) & (df[COL_B] == model5)) |
    ((df[COL_A] == model5) & (df[COL_B] == model3))
)
df_pair = df[mask].copy()
n_obs   = len(df_pair)

print(f"\n── Vérification ──")
print(f"Paire          : {model3}  vs  {model5}")
print(f"Comparaisons directes observées : {n_obs}")
print(f"Minimum requis (puissance 80%)  : {n_min}")

if n_obs == 0:
    print("\n✗ Ces deux modèles ne se sont jamais affrontés directement dans les données.")
    print("  → Impossible de conclure sur cette paire par test binomial direct.")
    print("  → C'est précisément pourquoi le modèle Bradley-Terry est utile :")
    print("    il estime les β via les affrontements indirects (transitifs).")
    print(f"  → Selon le modèle : P({model3} bat {model5}) = {p_ij:.3f}")
elif n_obs >= n_min:
    wins_3 = (df_pair[COL_WINNER] == model3).sum()
    wins_5 = (df_pair[COL_WINNER] == model5).sum()
    ties   = df_pair["both_equal"].fillna(False).astype(bool).sum()
    print(f"\n✓ Condition satisfaite ({n_obs} ≥ {n_min})")
    print(f"  Victoires {model3:<30} : {wins_3}")
    print(f"  Victoires {model5:<30} : {wins_5}")
    print(f"  Ex-æquo                              : {ties}")
else:
    print(f"\n✗ Données insuffisantes ({n_obs} < {n_min} comparaisons requises)")

Valeurs uniques de chosen_model_name : chosen_model_name
gemma-3-4b                    33
gemini-2.0-flash              24
gemini-1.5-pro                22
ministral-8b-instruct-2410    20
command-a                     18
gemma-3-12b                   17
mistral-medium-2508           15
llama-3.3-70b                 15
llama-3.1-405b                15
deepseek-v3-0324              15
Name: count, dtype: int64

Nombre de votes dans le top20 : 221

Classement global (top 20) :
 rank                           model      beta
    1              mistral-large-2411  1.116384
    2                      gemma-3-4b  0.595431
    3             mistral-medium-2508  0.470319
    4                       command-a  0.223354
    5                   llama-4-scout  0.004431
    6                 aya-expanse-32b  0.000000
    7                gemini-2.0-flash -0.030374
    8            claude-3-5-sonnet-v2 -0.052334
    9                     gemma-3-27b -0.273061
   10                     gemma-3-12b -0

Sur l'explication de la transitivité
Le modèle Bradley-Terry résout exactement le problème que tu viens d'observer. Rang 3 et rang 5 ne se sont jamais affrontés, mais ils ont tous les deux joué contre d'autres modèles. 

Par exemple :
mistral-medium-2508  a battu  gpt-5          (3 fois)

llama-4-scout        a perdu  mistral-saba   (2 fois)

gpt-5                a battu  llama-4-scout  (hypothétiquement)


Le modèle exploite ces chaînes indirectes pour estimer un β global cohérent. La MLE trouve les β qui maximisent la vraisemblance de tous les résultats observés simultanément — y compris les chemins qui passent par des modèles intermédiaires. Donc p35 = σ(β₃ − β₅) est une estimation valide même sans match direct, c'est précisément la force du modèle.


Le calcul de puissance répond alors à la question : "si on voulait confirmer cette estimation par un test direct, combien de matchs faudrait-il ?" — et la réponse est n_min, que les données n'atteignent pas. Ce n'est pas un bug, c'est une conclusion.